<a href="https://colab.research.google.com/github/mk654/SML_PG60/blob/main/COMP90051_ProjectGroup60_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **COMP90051 Group Project → Code**

| Project Group 60 |         |
|------------------|---------|
| Lachlan Fox      | 649622  |
| Songhao Guo      | 1542657 |
| Amelia King      | 1175861 |


github → https://github.com/mk654/SML_PG60


In [ ]:
# RUN FIRST
from scipy.io import loadmat
import pandas as pd
from pathlib import Path
import numpy as np
from scipy import sparse


# Amelia -> load data from github repo
!git clone https://github.com/mk654/SML_PG60
repo_dir = Path("/content/SML_PG60")

flu_mat = repo_dir / "data" / "matraw" / "influenza_outbreak_dataset.mat" # access influenza dataset from gitrepo
fludata = loadmat(flu_mat)



# url = "https://raw.githubusercontent.com/mk654/SML_PG60/main/influenza_outbreak_dataset.mat" # old access -> directory = main
#url = "https://raw.githubusercontent.com/mk654/SML_PG60/main/data/influenza_outbreak_dataset.mat" # old access -> directory = main/data


Cloning into 'SML_PG60'...
remote: Enumerating objects: 276, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 276 (delta 63), reused 1 (delta 1), pack-reused 159 (from 1)
Receiving objects: 100% (276/276), 41.65 MiB | 14.49 MiB/s, done.
Resolving deltas: 100% (112/112), done.


### Convert ```influenza_outbreak_dataset.mat``` to .csv
*influenza_outbreak_dataset.mat contains 48 folds (test/train splits). Each fold contains its own X & y train and X & y test:*
<div style="font-size: 0.75em;">

| name     | outer dtype | outer shape | inner type | inner shape |
|----------|-------------|-------------|------------|-------------|
| X train  | object      | (1, 48)     | csc_matrix | (1095, 545) |
| X test   | object      | (1, 48)     | csc_matrix | (485, 545)  |
| y train  | object      | (1, 48)     | ndarray    | (1095, 1)   |
| y test   | object      | (1, 48)     | ndarray    | (485, 1)    |
| locs     | object      | (1, 48)     | ndarray    | (1,)        |
| keywords | object      | (1, 525)    | ndarray    | (1,)        |

</div>

---

For further inspection/analysis, conversion produces 48 separate folders aligning with 48 folds within ```influenza_outbreak_dataset.mat```. Each folder contains:
  1. X_train.csv
  2. X_test.csv
  3. y_train.csv
  4. y_test.csv

Additionally, the conversion also produces:
  1. keywords.csv
  2. locs.csv

In [ ]:
# Amelia -> convert influenza_outbreak_dataset.mat to .csv file
flu_out = repo_dir / "data" / "processed" / "flu_csv" / "ak_flucsv"
flu_out.mkdir(parents=True, exist_ok=True)

X_tr = fludata["flu_X_tr"]
X_te = fludata["flu_X_te"]
y_tr = fludata["flu_Y_tr"]
y_te = fludata["flu_Y_te"]

n_folds = X_tr.shape[1]

for i in range(n_folds):
    fold_dir = flu_out / f"fold_{i:02d}"
    fold_dir.mkdir(exist_ok=True)

    Xtr = X_tr[0, i].toarray() # some data stored as sparse matrix, convert to dense
    Xte = X_te[0, i].toarray()
    ytr = y_tr[0, i].ravel()
    yte = y_te[0, i].ravel()

    pd.DataFrame(Xtr).to_csv(fold_dir / "X_train.csv", index=False)
    pd.DataFrame(Xte).to_csv(fold_dir / "X_test.csv", index=False)
    pd.DataFrame(ytr).to_csv(fold_dir / "y_train.csv", index=False)
    pd.DataFrame(yte).to_csv(fold_dir / "y_test.csv", index=False)

keywords = fludata["flu_keywords"]
keywords_list = [str(k[0]) for k in keywords.ravel()]
pd.DataFrame(keywords_list, columns=["keyword"]).to_csv(flu_out / "keywords.csv", index=False)

locs = fludata["flu_locs"]
locs_list = [str(l[0]) for l in locs.ravel()]
pd.DataFrame(locs_list, columns=["location"]).to_csv(flu_out / "locs.csv", index=False)

combines all information from influenza_outbreak_dataset.mat into  ```flu_long.csv``` (~168.6 MB)
- original X matrices within .mat file contain 545 features, however, only 525 named keyword features exist
  * thus ```flu_long.csv``` drops last 20 features within original X matrices
- drops original test/train split from ```influenza_outbreeak_dataset.mat```

In [ ]:
# Songhao -> combine all folds into one long-format .csv
combined_out = Path(dir / "data/processed")
combined_out.mkdir(parents=True, exist_ok=True)
#flu_out = Path(flucsv_dir/ "ak_flucsv")
keywords = pd.read_csv(flu_out / "keywords.csv")["keyword"].tolist()
locs = pd.read_csv(flu_out / "locs.csv")["location"].tolist()

all_parts = []

for i, loc in enumerate(locs):
    fold_dir = flu_out / f"fold_{i:02d}"

    X_train = pd.read_csv(fold_dir / "X_train.csv")
    y_train = pd.read_csv(fold_dir / "y_train.csv")
    X_test = pd.read_csv(fold_dir / "X_test.csv")
    y_test = pd.read_csv(fold_dir / "y_test.csv")

    X = pd.concat([X_train, X_test], ignore_index=True)
    y = pd.concat([y_train, y_test], ignore_index=True)
    X= np.asarray(X)
    X = X[:, :len(keywords)] # keep only keyword features
    y = np.asarray(y).reshape(-1).astype(int)

    df_part = pd.DataFrame(X, columns=keywords)
    df_part.insert(0, "location", loc)

    df_part["label"] = y

    all_parts.append(df_part)
df = pd.concat(all_parts, ignore_index=True)
csv_output_path = combined_out / "flu_long.csv"
df.to_csv(csv_output_path, index=False)

print("flu_long shape:", df.shape)
df.head()

### covid19_tweets.csv preprocesing
1. Filter out instances with:
    - empty location values
    - locations outside of the US

In [ ]:
#print(c_df.columns)
#print(c_df["user_location"].head())
import re

c_df["user_location"] = c_df["user_location"].fillna("").astype(str).str.strip().str.lower()
# dictionaries
abbr_to_state = {
    "al": "alabama","ak": "alaska", "az": "arizona", "ar": "arkansas", "ca": "california","co": "colorado",
    "ct": "connecticut", "de": "delaware", "fl": "florida", "ga": "georgia", "hi": "hawaii", "id": "idaho",
    "il": "illinois", "in": "indiana", "ia": "iowa", "ks": "kansas", "ky": "kentucky", "la": "louisiana",
    "me": "maine", "md": "maryland", "ma": "massachusetts", "mi": "michigan", "mn": "minnesota",
    "ms": "mississippi", "mo": "missouri", "mt": "montana", "ne": "nebraska", "nv": "nevada",
    "nh": "new hampshire", "nj": "new jersey", "nm": "new mexico", "ny": "new york",
    "nc": "north carolina", "nd": "north dakota", "oh": "ohio", "ok": "oklahoma", "or": "oregon",
    "pa": "pennsylvania", "ri": "rhode island", "sc": "south carolina", "sd": "south dakota",
    "tn": "tennessee", "tx": "texas", "ut": "utah", "vt": "vermont", "va": "virginia",
    "wa": "washington", "wv": "west virginia", "wi": "wisconsin", "wy": "wyoming", "dc": "district of columbia"
}
state_names = set(abbr_to_state.values())

usa_terms = {"usa", "us", "united states", "america", "u.s.", "u.s.a."}

city_to_state = {
    "new york": "new york", "nyc": "new york", "brooklyn": "new york", "manhattan": "new york",
    "los angeles": "california", "san diego": "california", "san francisco": "california", "sacramento": "california", "long beach": "california",
    "houston": "texas", "dallas": "texas", "austin": "texas",
    "miami": "florida", "orlando": "florida",
    "chicago": "illinois",
    "atlanta": "georgia",
    "boston": "massachusetts",
    "las vegas": "nevada",
    "seattle": "washington",
    "new orleans": "louisiana",
    "st louis": "missouri", "saint louis": "missouri",
    "washington dc": "district of columbia"
}

# specific non-us locations/terms to exclude
junk = ["everywhere", "worldwide", "23 countries", "opt-out", "catch me", "the beach",
        "in the vineyard", "available now", "working"]

foreign = ["canada", "uk", "australia", "india", "germany", "france", "italy", "spain", "brazil",
           "argentina", "china", "japan", "south korea", "paris", "london", "vienna", "delhi", "rio",
           "beijing", "nairobi", "joburg"]

bad_ab = {"in", "or", "me", "hi"}

def is_ambig(loc): # multiple locations or terms foreign to usa
    separators = [",", ";", "|", "/", " and ", " & "]
    foreign_count = 0
    for term in foreign:
        if term in loc:
            foreign_count += 1
    if foreign_count >= 1 and any (sep in loc for sep in separators):
        return True
    return False

def is_junk(loc):
    for term in junk:
        if term in loc:
            return True
    return False

#filter
def get_state(loc):
    loc = loc.lower().strip()

    if loc == "":
        return None

    if is_junk(loc):
        return None

    if is_ambig(loc):
        return None

    for state in state_names:
        if state in loc:
            return state

    tokens = re.findall(r"\b[a-z]{2}\b", loc)
    for token in tokens:
        if token in abbr_to_state and token not in bad_ab:
            return abbr_to_state[token]

    for city in city_to_state:
        if city in loc:
            return city_to_state[city]
    return None

2. evaluate confidence levels for state disambiguation for filtered c19 twitter data

*millie note: keep? hmm...*

In [ ]:
# amelia -> confidence levels for location extraction (high, medium, low, none)
def loc_conf(loc):
    loc = loc.lower().strip()
    for state in state_names:
        if state in loc:
            return "high"
    tokens = re.findall(r"\b[a-z]{2}\b", loc)
    for token in tokens:
        if token in abbr_to_state and token not in bad_ab:
            return "high"
    for city in city_to_state:
        if city in loc:
            return "medium"
    for term in usa_terms:
        if term in loc:
            return "low"
    return "none"

c_df["state"] = c_df["user_location"].apply(get_state)
c_df["state_confidence"] = c_df["user_location"].apply(loc_conf)
state_cdf = c_df[c_df["state"].notna()].copy()
print(state_cdf[["user_location", "state", "state_confidence"]].head(100))
print(len(state_cdf), "rows with identified US state locations")

#state_cdf[["user_location", "state", "state_confidence"]].to_csv(dir/"data/processed/c19_loc_review.csv", index=False)

3. remove irrelvant features:
    - ```user_name```
    - ```user_description```
    - ```user_created```
    - ```user_followers``` KEEP???
    - ```user_friends``` KEEP????
    - ```user_favourites```
    - ```user_verfied```
    - ```source``` KEEP????
    - ```is_retweet```

In [ ]:
# amelia -> remove irrelevant feature columns from covid19_tweets.csv
columns_to_keep = ["state", "date", "text", "hashtags"]
cdf_reduced = (c_df[c_df["state"].notna()][columns_to_keep].copy())
#cdf_reduced.to_csv(dir / "data/processed/covid19_twts.csv", index=False)
print(cdf_reduced.head())
print(cdf_reduced.shape)

4. convert tweet text to dictionary using ```flu_long.csv``` (from influenza_outbreak_dataset.mat); apply
    - this creates the covid dataset 1, wherein

In [ ]:
# Lachlan -> vectorise tweets based on flu keywords
flu_data = Path(dir / "data/processed/flu_long.csv")
flu_df= pd.read_csv(flu_data)
flu_columns = flu_df.columns.tolist()
columns_to_drop = ['location', 'label']
flu_keywords = [item for item in flu_columns if item not in columns_to_drop]

def vectorise_tweet(tweet):
  text = tweet.lower()
  counts = {}
  for keyword in flu_keywords:
    pattern = r'\b' + re.escape(keyword) + r'\b'
    counts[keyword] = len(re.findall(pattern, text))
  return counts

# Amelia -> apply
covid_vecs = cdf_reduced["text"].apply(vectorise_tweet)
covid_vecs_df = pd.DataFrame(covid_vecs.tolist())
covid19_vectorised = pd.concat([cdf_reduced.reset_index(drop=True), covid_vecs_df.reset_index(drop=True)], axis=1)
#covid19_vectorised = pd.concat([cdf_reduced[["state", "date", "text", "hashtags"]].reset_index(drop=True),
 #                               covid_vecs_df.reset_index(drop=True)], axis=1) #from when I was applying to a csv that dont exist no mo :-]

covid19_vectorised.to_csv(dir / "data/processed/covid19_twts_vectorised.csv", index=False)

5. create second ```covid19_tweets.csv``` derived dataset wherein own tweet keywords are created
    - clean tweet text first

In [ ]:
# Amelia -> clean tweet text (remove urls, mentions, hashtags, punctuation, extra whitespace)
def clean_tweet(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text
cdf_reduced["clean_text"] = cdf_reduced["text"].apply(clean_tweet)
print(cdf_reduced[["text", "clean_text"]].head())

tf-idf

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer # do I have to do this from scratch...
tfidf = TfidfVectorizer(max_features=1000, stop_words='english', ngram_range=(1,2), min_df=5)
X_covid = tfidf.fit_transform(cdf_reduced["clean_text"])

covid_keywords = tfidf.get_feature_names_out()
c_tfidf_df = pd.DataFrame(X_covid.toarray(), columns=covid_keywords)

covid19_twts = pd.concat([cdf_reduced.reset_index(drop=True), c_tfidf_df.reset_index(drop=True)], axis=1) #need to double check this output!
covid19_twts= covid19_twts.rename(columns={"state":"location"})

covid19_twts.to_csv(dir / "data/processed/covid19_twts_COPY.csv", index=False)

need to edit this so it works with my files

In [ ]:
#Lachlan -> Sort COVID tweets by whether or not the 'covid19' hashtag appears.

covid_df = pd.read_csv('covid19_tweets.csv')
word_count_series = covid_df['text'].apply(vectorise_tweet)
covid_count_df = pd.DataFrame(list(word_count_series))
covid_vector_df = covid_count_df.reindex(columns = flu_df.columns, fill_value = 0)



In [ ]:
def hashtag_check(hashtags):
  lower = hashtags.lower()
  targets = ['covid19', 'coronavirus']
  if any(item in lower for item in targets):
    return 1
  else:
    return 0

hashtag_series = covid_df['hashtags'].copy()
hashtag_series = hashtag_series.fillna("")

covid_vector_df['label'] = hashtag_series.apply(hashtag_check)
covid_vector_df.head()

,location,split,time_index,flu,swine,stomach,symptoms,virus,bug,strep,...,tests,thinks,ankle,work,hand,complications,children,start,aja,label
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,1


### lachlan training (replace title later)

In [ ]:
#Lachlan -> parameter metrics
import torch
import torch.nn as nn

def accuracy_score(preds, y):
  correct = (preds == y).sum()
  return correct / len(y)

def precision_score(preds, y): #preds and y are arrays of 0 and 1
  tp = (preds * y).sum() #Only indicies where preds and y are 1 will be counted
  pred_positives = (preds == 1).sum()
  return tp / (pred_positives + 1e-7) #To prevent div 0 problems

def recall_score(preds, y): #preds and y are arrays of 0 and 1
  tp = (preds * y).sum() #Only indicies where preds and y are 1 will be counted
  real_positives = (y == 1).sum()
  return tp / (real_positives + 1e-7) #To prevent div 0 problems

def f1_score(preds, y):
    prec = precision_score(preds, y)
    rec = recall_score(preds, y)
    return 2 * (prec * rec) / (prec + rec + 1e-7)



In [ ]:
#Lachlan -> basic logistic regression model
class LogisticRegressionModel(nn.Module): #From tute
    def __init__(self, input_dim):
        super(LogisticRegressionModel, self).__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.linear(x)

def train_model(model_class, input_dim, criterion_fn, lr, momentum, X_train, y_train, epochs=50, batch_size=32):
    model = model_class(input_dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    #Convert training and evaluation arrays to tensors
    Xt = torch.tensor(X_train, dtype = torch.float32)
    yt = torch.tensor(y_train, dtype = torch.float32).unsqueeze(1)


    #Set up training data loader
    dataset = torch.utils.data.TensorDataset(Xt, yt)
    train_loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle= True)

    for epoch in range(epochs):
      model.train()
      for batch_X, batch_y in train_loader:
          optimizer.zero_grad()
          predictions = model(batch_X)
          loss = criterion_fn(predictions, batch_y)
          loss.backward()
          optimizer.step()

    return model

# Songhao -> stratified folds from scratch
def make_stratified_folds(y, n_folds, seed=42):
  rng = np.random.default_rng(seed)
  y = np.asarray(y).astype(int)

  folds = [[] for _ in range(n_folds)]

  for label in np.unique(y):
      label_indices = np.where(y == label)[0]
      rng.shuffle(label_indices)

      for i, idx in enumerate(label_indices):
          folds[i % n_folds].append(idx)

  return [np.array(fold, dtype=int) for fold in folds]

# Amelia -> implement like skleasrn import
def stratified_split_indices(X, y, n_folds, seed=42):
  """uses Songhao's make_stratified_folds function ot generate (train_idx, val_idx)
  similar to StratifiedKFold.split(X,y)"""
  val_folds= make_stratified_folds(y, n_folds, seed)
  all_indices= np.arange(len(y))
  for val_idx in val_folds:
    train_idx=np.setdff1d(all_indices, val_idx)
    yield train_idx, val_idx

In [ ]:
#Lachlan -> Cross-validated logistic regression
#from sklearn.model_selection import StratifiedKFold


#Inputs are np arrays X and y
def cross_val(k_folds, X, y, model, criterion, lr = 0.001, momentum = 1e-2):
  n_features = X.shape[1]
  #skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
  fold_accuracies = []
  fold_precisions = []
  fold_recalls = []
  fold_f1 = []
  best_f1 = -1.0
  best_model = None

  #folds = skf.split(X, y) #Create training, validation and testing folds
  folds= stratified_split_indices(X, y, k_folds, seed=42)

  for fold, (train_idx, val_idx) in enumerate(folds):
      X_train, X_eval = X[train_idx], X[val_idx]
      y_train, y_eval = y[train_idx], y[val_idx]
      Xe = torch.tensor(X_eval, dtype = torch.float32)

      fold_model = train_model(model, n_features, criterion, lr, momentum, X_train, y_train, epochs=50)

      fold_model.eval()
      with torch.no_grad():
        outputs = fold_model(Xe) #Determine the raw probabilities
        preds = (torch.sigmoid(outputs) >= 0.5).float().numpy().flatten() #Convert to array of 0 and 1

      acc = accuracy_score(preds, y_eval)
      prec = precision_score(preds, y_eval)
      rec = recall_score(preds, y_eval)
      f1 = f1_score(preds, y_eval)
      fold_accuracies.append(acc)
      fold_precisions.append(prec)
      fold_recalls.append(rec)
      fold_f1.append(f1)
      if f1 > best_f1:
        best_f1 = f1
        best_model = fold_model

  avg_acc = np.mean(fold_accuracies)
  avg_prec = np.mean(fold_precisions)
  avg_rec = np.mean(fold_recalls)
  avg_f1 = np.mean(fold_f1)

  return avg_acc, avg_prec, avg_rec, avg_f1, best_model


In [ ]:
#Lachlan -> run baseline logistic regression

from sklearn.preprocessing import StandardScaler
df = pd.read_csv("flu_long.csv")
df.drop(columns = ['location','split', 'time_index'], inplace = True) #Basic version
df['label'] = df['label'].astype(int) #Cast entries as ints

X_np = (df.drop(columns = ['label'])).values
y_np = df['label'].values


criterion = torch.nn.BCEWithLogitsLoss()

#Configure for cross-validation
k_folds = 10

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_np)
avg_acc,  avg_prec, avg_rec, avg_f1, best_model = cross_val(k_folds, X_scaled, y_np, LogisticRegressionModel, criterion, 0.001, 0.01)

print(f"Averages: Accuracy {avg_acc} | Precision {avg_prec} | Recall {avg_rec} | F1 {avg_f1}")

In [ ]:
#Lachlan -> keyword counting
flu_df = pd.read_csv("flu_long.csv")
flu_columns = flu_df.columns.tolist()
columns_to_drop = ['location', 'split', 'time_index', 'label']
flu_keywords = [item for item in flu_columns if item not in columns_to_drop]

keyword_totals = {}
for keyword in flu_keywords:
  total = flu_df[keyword].sum()
  keyword_totals[keyword] = total

sorted_keywords = sorted(keyword_totals, key = keyword_totals.get)

['type', 'ive', 'stupid', 'sweating', 'activity', 'forward', 'slowly', 'nap', 'definitely', 'yay', 'helped', 'either', 'hospitalized', 'heard', 'healing', 'wasn', 'thru', 'loss', 'illnesses', 'waiting', 'knocked', 'decided', 'past', 'lost', 'normal', 'couldn', 'shift', 'ish', 'stuff', 'drinking', 'bring', 'kind', 'holiday', 'lose', 'mist', 'eaten', 'rather', 'spent', 'wanted', 'aku', 'bottle', 'wants', 'survived', 'welcome', 'fast', 'pra', 'wonderful', 'appetite', 'drop', 'birds', 'goodness', 'giving', 'thinks', 'start', 'managed', 'freaking', 'aids', '101', 'banget', 'achy', 'miss', 'wouldn', 'dad', 'kalo', 'currently', 'bro', 'likely', 'quite', 'dose', 'pretty', 'doc', '24hr', 'super', 'quarantined', 'players', 'bgt', 'classes', '2nd', 'enough', 'everywhere', 'medication', 'sama', 'entire', 'care', 'infections', 'apparently', 'swear', 'episode', 'lucky', 'juice', 'hand', 'dealing', 'sanitizer', 'beat', 'viruses', 'crappy', 'passed', 'started', 'officially', 'wear', 'late', 'onset', '

In [ ]:
#Lachlan -> reduction by dropping columns
num_keys = len(sorted_keywords)
x = 100 #Number of desired keywords
num_to_drop = num_keys - x
columns_to_drop = sorted_keywords[:num_to_drop]
reduced_flu_df = flu_df.drop(columns = columns_to_drop)
reduced_flu_df.head(5)


,location,split,time_index,flu,swine,stomach,symptoms,virus,bug,strep,...,related,panas,pox,diarrhea,tonsillitis,killer,stayed,immunity,dehydration,label
0,wyoming,train,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,wyoming,train,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,wyoming,train,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,wyoming,train,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,wyoming,train,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [ ]:
#Lachlan -> autoencoder

class FeatureAutoencoder(nn.Module):
    def __init__(self, input_dim, bottleneck_dim):
        super().__init__()

        hidden_dim = max(256, bottleneck_dim * 2)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, bottleneck_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

def encode_data(X, new_dim, epochs = 50, batch_size = 32, lr = 0.001):
  #Convert to tensor
  Xt = torch.tensor(X, dtype = torch.float32)
  X_dim = Xt.shape[1]

  dataset = torch.utils.data.TensorDataset(Xt)
  data_loader = torch.utils.data.DataLoader(dataset, batch_size = batch_size, shuffle = True)

  autoencoder_model = FeatureAutoencoder(input_dim = X_dim, bottleneck_dim = new_dim)
  criterion = nn.MSELoss()
  #Use Adam for best performance
  optimizer = torch.optim.Adam(autoencoder_model.parameters(), lr = lr)

  autoencoder_model.train()
  for i in range(epochs):
    for (batch_x,) in data_loader:
      reconstructed_data = autoencoder_model(batch_x)
      loss = criterion(reconstructed_data, batch_x)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

  autoencoder = autoencoder_model.encoder
  autoencoder.eval()
  with torch.no_grad():
    reduced_X = autoencoder(Xt)

  new_X_np = reduced_X.detach().cpu().numpy()

  return new_X_np

In [ ]:
#Lachlan -> Determine optimal autoencoding

#X and y must be numpy arrays
#X must be 2D

def opt_autoencode(k_folds,step_size, X, y, epochs = 50, batch_size = 32, lr = 0.001, momentum = 1e-2):
  X_len = X.shape[1] #Starting length
  test_len = np.floor_divide(X_len, 2)
  best_f1 = 0
  best_len = 0
  best_X = np.empty(X.shape)

  while test_len > 10:
    X_encoded = encode_data(X, test_len, epochs = epochs, batch_size= batch_size, lr = lr)
    criterion = torch.nn.BCEWithLogitsLoss()
    test_f1 = (cross_val(k_folds, X_encoded, y, LogisticRegressionModel,criterion, lr, momentum))[3]
    if test_f1 > best_f1:
      best_len = test_len
      best_X = X_encoded.copy()
    test_len = test_len - 50

  return best_X


In [ ]:
#Lachlan -> Autoencode X
from sklearn.preprocessing import StandardScaler
df = pd.read_csv("flu_long.csv")
df.drop(columns = ['location'], inplace = True) #Basic version
df['label'] = df['label'].astype(int) #Cast entries as ints

scaler = StandardScaler()
X_np = scaler.fit_transform((df.drop(columns = ['label'])).values)
y_np = df['label'].values
k_folds = 10
step_size = 50
epochs = 30
batch_size = 32
lr = 0.001
momentum = 1e-2


X_reduced = opt_autoencode(k_folds, step_size, X_np, y_np, epochs, batch_size, lr, momentum)
print(X_reduced.shape)

## CatBoost
This section adds CatBoost as the complex model. The artificial ```time_index``` / ```timestamp``` columns are removed because they were added during preprocessing and do not represent real temporal information.

In [ ]:
# Songhao -> CatBoost model
!pip install catboost
from catboost import CatBoostClassifier

In [ ]:
# Songhao -> Prepare features and labels

target_col = "label"

drop_cols = ["location"]
drop_cols = [col for col in drop_cols if col in df.columns]

keyword_cols = [
    col for col in df.columns
    if col not in drop_cols + [target_col]
]

X_raw = df[keyword_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
y = df[target_col].astype(int).to_numpy()

# Non-temporal feature construction
X = X_raw.copy()
X["keyword_total_intensity"] = X_raw.sum(axis=1)
X["active_keyword_count"] = (X_raw > 0).sum(axis=1)
X["nonzero_keyword_ratio"] = X["active_keyword_count"] / len(keyword_cols)
X["max_keyword_value"] = X_raw.max(axis=1)
X["mean_keyword_value"] = X_raw.mean(axis=1)
X["std_keyword_value"] = X_raw.std(axis=1)
X["log_total_intensity"] = np.log1p(X["keyword_total_intensity"])

catboost_feature_columns = X.columns.tolist()

print("Dropped columns:", drop_cols)
print("Original keyword features:", len(keyword_cols))
print("Final feature shape:", X.shape)
print("Label distribution:")
print(pd.Series(y).value_counts())

In [ ]:
#  Songhao -> Metrics and stratified k-fold cross-validation from scratch

def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


# commented out for now; code for this is above in logistic regression cell
# should probably put common functions for cross-val and so on at the start of the notebook for easy access
"""def make_stratified_folds(y, n_folds, seed=42):
  rng = np.random.default_rng(seed)
  y = np.asarray(y).astype(int)

  folds = [[] for _ in range(n_folds)]

  for label in np.unique(y):
      label_indices = np.where(y == label)[0]
      rng.shuffle(label_indices)

      for i, idx in enumerate(label_indices):
          folds[i % n_folds].append(idx)

  return [np.array(fold, dtype=int) for fold in folds] """


def get_train_indices(n_samples, test_indices):
    all_indices = np.arange(n_samples)
    return np.setdiff1d(all_indices, test_indices)

def summarise_results(results_df):
    summary_rows = []

    for metric in ["accuracy", "precision", "recall", "f1"]:
        values = results_df[metric].to_numpy(dtype=float)

        summary_rows.append({
            "metric": metric,
            "mean": values.mean(),
            "std": values.std(ddof=1),
        })

    return pd.DataFrame(summary_rows)

In [ ]:
# Songhao -> CatBoost with nested cross-validation
def train_catboost(X_train, y_train, depth, iterations=200, learning_rate=0.05, seed=42):
    model = CatBoostClassifier(
        iterations=iterations,
        depth=depth,
        learning_rate=learning_rate,
        loss_function="Logloss",
        eval_metric="F1",
        auto_class_weights="Balanced",
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
        task_type=CATBOOST_TASK_TYPE,
        devices="0" if CATBOOST_TASK_TYPE == "GPU" else None,
    )

    model.fit(X_train, y_train)
    return model

def tune_catboost_depth(X_train_outer, y_train_outer, depth_grid, inner_folds=3, seed=42):
    inner_fold_indices = make_stratified_folds(
        y_train_outer,
        n_folds=inner_folds,
        seed=seed,
    )

    tuning_results = []

    for depth in depth_grid:
        inner_scores = []

        for inner_fold_id, val_idx in enumerate(inner_fold_indices):
            train_idx = get_train_indices(len(y_train_outer), val_idx)

            X_inner_train = X_train_outer.iloc[train_idx]
            y_inner_train = y_train_outer[train_idx]
            X_inner_val = X_train_outer.iloc[val_idx]
            y_inner_val = y_train_outer[val_idx]

            model = train_catboost(
                X_inner_train,
                y_inner_train,
                depth=depth,
                iterations=120,
                learning_rate=0.05,
                seed=seed + inner_fold_id,
            )

            y_val_pred = model.predict(X_inner_val).astype(int)
            val_metrics = compute_metrics(y_inner_val, y_val_pred)
            inner_scores.append(val_metrics["f1"])

        tuning_results.append({
            "depth": depth,
            "mean_inner_f1": float(np.mean(inner_scores)),
            "std_inner_f1": float(np.std(inner_scores, ddof=1)),
        })

    tuning_df = pd.DataFrame(tuning_results)
    best_depth = int(
        tuning_df.sort_values("mean_inner_f1", ascending=False).iloc[0]["depth"]
    )

    return best_depth, tuning_df

outer_folds = make_stratified_folds(y, n_folds=10, seed=42)
depth_grid = [10, 12, 14]

catboost_outer_results = []
catboost_tuning_logs = []

for outer_fold_id, test_idx in enumerate(outer_folds, start=1):
    print(f"Running outer fold {outer_fold_id}/10")

    train_idx = get_train_indices(len(y), test_idx)

    X_outer_train = X.iloc[train_idx].reset_index(drop=True)
    y_outer_train = y[train_idx]
    X_outer_test = X.iloc[test_idx].reset_index(drop=True)
    y_outer_test = y[test_idx]

    best_depth, tuning_df = tune_catboost_depth(
        X_outer_train,
        y_outer_train,
        depth_grid=depth_grid,
        inner_folds=3,
        seed=100 + outer_fold_id,
    )

    tuning_df["outer_fold"] = outer_fold_id
    catboost_tuning_logs.append(tuning_df)

    final_model = train_catboost(
        X_outer_train,
        y_outer_train,
        depth=best_depth,
        iterations=200,
        learning_rate=0.05,
        seed=200 + outer_fold_id,
    )

    y_test_pred = final_model.predict(X_outer_test).astype(int)
    test_metrics = compute_metrics(y_outer_test, y_test_pred)

    test_metrics["outer_fold"] = outer_fold_id
    test_metrics["best_depth"] = best_depth

    catboost_outer_results.append(test_metrics)

catboost_results_df = pd.DataFrame(catboost_outer_results)
catboost_tuning_df = pd.concat(catboost_tuning_logs, ignore_index=True)
catboost_summary_df = summarise_results(catboost_results_df)

display(catboost_results_df)
display(catboost_summary_df)


In [ ]:
# Songhao -> Train final CatBoost model on the full influenza dataset

best_depth_overall = int(catboost_results_df["best_depth"].mode().iloc[0])

final_catboost_model = train_catboost(
    X,
    y,
    depth=best_depth_overall,
    iterations=200,
    learning_rate=0.05,
    seed=999,
)

print("Final CatBoost model trained.")
print("Selected depth:", best_depth_overall)

In [ ]:

# Songhao -> Save CatBoost results

results_dir = repo_dir / "data" / "processed" / "catboost_results"
results_dir.mkdir(parents=True, exist_ok=True)

catboost_results_df.to_csv(results_dir / "catboost_outer_cv_results.csv", index=False)
catboost_tuning_df.to_csv(results_dir / "catboost_inner_tuning_results.csv", index=False)
catboost_summary_df.to_csv(results_dir / "catboost_summary.csv", index=False)

print("Saved CatBoost results to:", results_dir)